# Clase 042 — Mapas (folium / plotly)

**Parte 0** · folium + plotly docs.

> 🎯 Mapas básicos para datos con componente geográfico. Sin entrar a GIS profundo.

> ⏱️ ~60 min

## ⚙️ Setup

```bash
pip install folium plotly
```

In [ ]:
try:
    import folium
    import plotly.express as px
    print(f'folium: {folium.__version__}')
    print('plotly OK')
except ImportError as e:
    print(f'Instala dependencias: pip install folium plotly  ({e})')

## 1️⃣ folium — mapa con markers

In [ ]:
ciudades = [
    {'nombre': 'Madrid',    'lat': 40.4168, 'lng': -3.7038, 'pop': 3_300_000},
    {'nombre': 'Barcelona', 'lat': 41.3851, 'lng':  2.1734, 'pop': 1_640_000},
    {'nombre': 'Valencia',  'lat': 39.4699, 'lng': -0.3763, 'pop':   800_000},
    {'nombre': 'Sevilla',   'lat': 37.3891, 'lng': -5.9845, 'pop':   690_000},
    {'nombre': 'Zaragoza',  'lat': 41.6488, 'lng': -0.8891, 'pop':   680_000},
]

m = folium.Map(location=[40, -4], zoom_start=6, tiles='OpenStreetMap')
for c in ciudades:
    color = 'green' if c['pop'] > 1_000_000 else ('orange' if c['pop'] > 700_000 else 'red')
    folium.CircleMarker(
        location=[c['lat'], c['lng']],
        radius=8,
        color=color, fill=True, fill_color=color, fill_opacity=0.7,
        popup=folium.Popup(f"<b>{c['nombre']}</b><br>pop: {c['pop']:,}", max_width=200),
        tooltip=c['nombre'],
    ).add_to(m)

m   # en notebook se renderiza inline

## 2️⃣ Convenciones de coordenadas

⚠️ **Cuidado**:
- **folium / Leaflet**: `[lat, lng]` (lat primero)
- **plotly / GeoJSON**: `[lng, lat]` (lng primero, estándar GeoJSON)

Este es el error #1 al hacer mapas.

## 3️⃣ plotly choropleth

Mapa de calor por país (o región) — plotly tiene shapes built-in para países usando códigos ISO-3:

In [ ]:
try:
    import pandas as pd
    rng = __import__('numpy').random.default_rng(42)
    
    paises_iso = ['ESP', 'FRA', 'DEU', 'ITA', 'PRT', 'GBR', 'POL', 'NLD', 'BEL', 'CHE']
    nombres    = ['España', 'Francia', 'Alemania', 'Italia', 'Portugal', 'Reino Unido', 'Polonia', 'Países Bajos', 'Bélgica', 'Suiza']
    valor      = rng.uniform(100, 500, len(paises_iso)).round(1)
    df = pd.DataFrame({'iso': paises_iso, 'nombre': nombres, 'valor': valor})
    
    fig = px.choropleth(
        df, locations='iso', color='valor', hover_name='nombre',
        color_continuous_scale='Viridis', scope='europe',
        title='Choropleth Europa (valor sintético)',
    )
    fig.show()
except ImportError:
    print('Instala plotly: pip install plotly')

## 4️⃣ folium choropleth con GeoJSON

```python
m = folium.Map(location=[0, 0], zoom_start=2)
folium.Choropleth(
    geo_data='https://...countries.geojson',   # GeoJSON con shapes
    name='choropleth',
    data=df,                                    # DataFrame con (codigo, valor)
    columns=['codigo', 'valor'],
    key_on='feature.properties.ISO3',          # ruta dentro del GeoJSON
    fill_color='YlGn',
    legend_name='valor',
).add_to(m)
```

Ventaja folium: mapa físico explorable (zoom, pan, popup). plotly: integra mejor con dashboards.

## 5️⃣ Tile providers

- **OpenStreetMap** (default) — colaborativo, gratis.
- **CartoDB Positron** — limpio y discreto.
- **CartoDB DarkMatter** — para dark themes.
- **Stamen Terrain** — relieve.
- **Mapbox** (require API key).

```python
folium.Map(tiles='CartoDB positron')
folium.TileLayer('CartoDB dark_matter').add_to(m)   # capa adicional
```

## 6️⃣ Cuándo geopandas

Esta clase cubre **mapas para visualización**. Si necesitas:
- Operaciones espaciales (intersection, buffer, dissolve)
- Re-proyecciones entre CRS
- Análisis raster

…usa **geopandas** (extiende pandas con geometrías Shapely). Fuera del scope de Parte 0; aparecería en un curso aparte de GIS.

## ✅ Checklist

- [ ] Sé crear mapa folium con markers y popups
- [ ] Conozco el bug lat/lng vs lng/lat
- [ ] Hago choropleth con folium o plotly
- [ ] Elijo tile provider según estética
- [ ] Sé cuándo necesito geopandas (no para esto)

## 📝 Homework

Ver `README.md`. Mapa con 10+ markers, choropleth folium y plotly, reporte comparativo.

## 📖 Definiciones y características

**Coordenadas geográficas (lat, lng)**

**Latitud** (-90 a 90): N/S del ecuador. **Longitud** (-180 a 180): E/W de Greenwich. Convención y orden varía: folium `[lat, lng]`, GeoJSON `[lng, lat]` — fuente clásica de bugs.

**folium**

Wrapper Python de Leaflet.js. Mapas interactivos (zoom, pan, popups) embebidos en notebook como HTML. Ideal para exploración.

**plotly geo**

Mapas en plotly (scatter_geo, choropleth). Bonitos, interactivos, integran con dashboards Plotly/Dash. Para choropleth de países usa códigos ISO-3.

**Choropleth**

Mapa de calor por región: color de cada polígono representa un valor (PIB, población, casos). Requiere GeoJSON con las shapes.

**Tile provider**

Servidor de "baldosas" (imágenes 256×256) que componen el mapa de fondo. OpenStreetMap (default), CartoDB (limpio), Mapbox (premium).

**geopandas**

Extensión de pandas con geometrías (Shapely). Para **análisis** espacial (intersection, buffer, dissolve), no solo visualizar. Fuera del scope Parte 0.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| Marcadores aparecen en medio del océano | Invertiste lat/lng. **Fix**: folium espera `[lat, lng]`. Para Madrid: `[40.42, -3.70]` (lat positiva, lng negativa). |
| Mapa folium no se renderiza en VS Code | Algunas extensiones de notebook no muestran HTML inline complejo. **Fix**: `m.save('mapa.html')` y abre en navegador. |
| choropleth plotly sale gris | El código ISO no matchea con el shape. **Fix**: usa ISO-3 (`'ESP'` no `'ES'`). Verifica que los códigos del dataset coinciden con los esperados. |
| Mapa pesa MB y carga lento | Demasiados markers (>1000). **Fix**: usa `MarkerCluster` (`folium.plugins.MarkerCluster`) que agrupa por zoom level. |
| Tiles fallan a cargar | Provider con rate limit (Mapbox sin token, Stamen down). **Fix**: usa `'OpenStreetMap'` o `'CartoDB positron'` que son gratis sin token. |

## ❓ Preguntas frecuentes

**❓ ¿folium o plotly?**

**folium** para exploración interactiva en notebook (mapa real con zoom/pan/popups). **plotly** para integrar en dashboard o cuando ya usas plotly. Ambos producen HTML standalone.

**❓ ¿Cómo encuentro lat/lng de un lugar?**

Click derecho en Google Maps → "¿Qué hay aquí?". O geocodifica con `geopy` (`from geopy.geocoders import Nominatim`).

**❓ ¿GeoJSON dónde lo consigo?**

Para países: [Natural Earth](https://www.naturalearthdata.com/) (público). Para regiones: portal gov del país. Para custom: dibuja en geojson.io.

**❓ ¿Cuándo necesito geopandas?**

Análisis espacial real: "¿cuántos puntos caen en este polígono?", "buffer de 1km alrededor", re-proyección entre CRS. Para solo visualizar puntos en mapa: folium basta.

**❓ ¿Mapas offline?**

Tiles son online por default. Para offline: descarga tiles con `mbtiles`, sirve local con `folium.raster_layers.TileLayer(url='file://...')`. Setup complejo, considera si vale.

## 🔗 Referencias

- [folium docs](https://python-visualization.github.io/folium/)
- [plotly choropleth](https://plotly.com/python/choropleth-maps/)

➡️ **Siguiente:** [041 — SQL fundamental](../041-sql-fundamental-select-where-join-group-by-having/README.md)